<a href="https://colab.research.google.com/github/aarushi-202618017/202618017_Aarushi_DS605/blob/main/202618017_LAB_04/202618017_LAB_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'new-york-city-airbnb-open-data' dataset.
Path to dataset files: /kaggle/input/new-york-city-airbnb-open-data


In [ ]:
import pandas as pd

df = pd.read_csv(path + "/AB_NYC_2019.csv")
print("Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
df.head()

Dataset Shape: (48895, 16)

Missing Values:
 id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [ ]:
import numpy as np
import pandas as pd

print("missing values")
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame(
    {"Missing Count": null_counts, "Percentage (%)": null_pct}
)
print(missing_df[missing_df["Missing Count"] > 0])


print("\noutliers\n")
cols_to_check = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
]
print(
    df[cols_to_check].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print("\ntarget value\n")
print(f"Total rows in raw dataset: {len(df)}")
print(f"Listings with price == 0: {(df['price'] == 0).sum()}")
print(f"Listings with price > $500: {(df['price'] > 500).sum()}")
print(f"Listings with price > $1,000: {(df['price'] > 1000).sum()}")
print(
    f"Listings with minimum_nights > 365: {(df['minimum_nights'] > 365).sum()}"
)

missing values
             Missing Count  Percentage (%)
last_review          10052       20.558339

outliers

              price  minimum_nights  number_of_reviews  reviews_per_month  \
count  48895.000000    48895.000000       48895.000000       48895.000000   
mean     152.720687        7.029962          23.274466           1.090910   
std      240.154170       20.510550          44.550582           1.597283   
min        0.000000        1.000000           0.000000           0.000000   
1%        30.000000        1.000000           0.000000           0.000000   
5%        40.000000        1.000000           0.000000           0.000000   
25%       69.000000        1.000000           1.000000           0.040000   
50%      106.000000        3.000000           5.000000           0.370000   
75%      175.000000        5.000000          24.000000           1.580000   
95%      355.000000       30.000000         114.000000           4.310000   
99%      799.000000       45.000000      

In [ ]:
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

df_clean = df[
    (df["price"] >= 10)
    & (df["price"] <= 500)
    & (df["minimum_nights"] <= 365)
].copy()

df_clean["dist_to_midtown"] = np.sqrt(
    (df_clean["latitude"] - 40.7580) ** 2
    + (df_clean["longitude"] - (-73.9855)) ** 2
)

num_features = [
    "latitude",
    "longitude",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
    "dist_to_midtown",
]
cat_features = ["neighbourhood_group", "room_type"]

X = df_clean[num_features + cat_features]
y = np.log1p(df_clean["price"])

print(f"Original dataset rows: {len(df)}")
print(f"Cleaned dataset rows:  {len(df_clean)}")
print(f"Total rows dropped:    {len(df) - len(df_clean)}")
print(f"\nMissing values remaining in X:\n{X.isnull().sum()}")

Original dataset rows: 48895
Cleaned dataset rows:  47826
Total rows dropped:    1069

Missing values remaining in X:
latitude                          0
longitude                         0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dist_to_midtown                   0
neighbourhood_group               0
room_type                         0
dtype: int64


# Task 2

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

# 1. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Preprocessor Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)

# 3. Candidate Models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest (Baseline)": RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
}

results = []

# 4. Evaluate Baseline Performance
for name, model in models.items():
    pipe = Pipeline(
        steps=[("preprocessor", preprocessor), ("regressor", model)]
    )
    pipe.fit(X_train, y_train)

    # Train and Test Predictions
    y_train_pred_log = pipe.predict(X_train)
    y_test_pred_log = pipe.predict(X_test)

    # Convert to USD scale
    y_train_orig, y_train_pred_orig = np.expm1(y_train), np.expm1(
        y_train_pred_log
    )
    y_test_orig, y_test_pred_orig = np.expm1(y_test), np.expm1(y_test_pred_log)

    # Calculate metrics
    results.append(
        {
            "Model": name,
            "Train R²": round(r2_score(y_train_orig, y_train_pred_orig), 4),
            "Test R²": round(r2_score(y_test_orig, y_test_pred_orig), 4),
            "Test RMSE ($)": round(
                np.sqrt(mean_squared_error(y_test_orig, y_test_pred_orig)), 2
            ),
            "Test MAE ($)": round(
                mean_absolute_error(y_test_orig, y_test_pred_orig), 2
            ),
        }
    )

comparison_df = pd.DataFrame(results).sort_values(
    by="Test R²", ascending=False
)
print(comparison_df.to_string(index=False))

                   Model  Train R²  Test R²  Test RMSE ($)  Test MAE ($)
Random Forest (Baseline)    0.5781   0.5068          62.71         39.05
           Decision Tree    0.5624   0.4647          65.33         41.15
       Linear Regression    0.4141   0.4180          68.12         42.86
        Ridge Regression    0.4140   0.4179          68.13         42.86


In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Hyperparameter Grid
param_dist = {
    "regressor__n_estimators": [100, 150, 200],
    "regressor__max_depth": [10, 15, 20],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf": [1, 2, 4],
}

# 2. Base Pipeline
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(random_state=42, n_jobs=-1),
        ),
    ]
)

# 3. Randomized Search Optimization
search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
)

print("Starting Hyperparameter Tuning...")
search.fit(X_train, y_train)

# 4. Best Model Evaluation
best_model = search.best_estimator_
y_test_pred_log = best_model.predict(X_test)

y_test_orig = np.expm1(y_test)
y_test_pred_orig = np.expm1(y_test_pred_log)

tuned_rmse = np.sqrt(mean_squared_error(y_test_orig, y_test_pred_orig))
tuned_mae = mean_absolute_error(y_test_orig, y_test_pred_orig)
tuned_r2 = r2_score(y_test_orig, y_test_pred_orig)

print("\nTUNED RANDOM FOREST RESULTS")
print(f"Best Hyperparameters: {search.best_params_}")
print(f"Tuned Test R²:       {tuned_r2:.4f}")
print(f"Tuned Test RMSE:     ${tuned_rmse:.2f}")
print(f"Tuned Test MAE:      ${tuned_mae:.2f}")

# 5. Save the Final Model Pipeline Artifact
joblib.dump(best_model, "airbnb_model.pkl")
print("\nFinal pipeline successfully saved as 'airbnb_model.pkl'")

Starting Hyperparameter Tuning...

TUNED RANDOM FOREST RESULTS
Best Hyperparameters: {'regressor__n_estimators': 150, 'regressor__min_samples_split': 5, 'regressor__min_samples_leaf': 1, 'regressor__max_depth': 15}
Tuned Test R²:       0.5219
Tuned Test RMSE:     $61.74
Tuned Test MAE:      $38.50

Final pipeline successfully saved as 'airbnb_model.pkl'


In [28]:
import joblib

# Re-save with compression level 3 (drastically reduces file size)
joblib.dump(best_model, "airbnb_model.pkl", compress=3)
print("Model saved with compression! Check the file size panel.")

Model saved with compression! Check the file size panel.


# Task 3

In [21]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.6 MB/s eta 0:00:00


In [32]:
%%writefile app.py
import os
import joblib
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(
    page_title="NYC Airbnb Price Predictor", layout="centered"
)

st.title("NYC Airbnb Nightly Price Predictor")
st.write("Estimate the optimal nightly price for an NYC listing.")


# Dynamically locate airbnb_model.pkl in the same directory as app.py
@st.cache_resource
def load_pipeline():
    base_dir = os.path.dirname(os.path.abspath(__file__))
    model_path = os.path.join(base_dir, "airbnb_model.pkl")
    return joblib.load(model_path)


pipeline = load_pipeline()

# Borough center coordinates mapping (eliminates forced coordinate input)
BOROUGH_COORDS = {
    "Manhattan": (40.7831, -73.9712),
    "Brooklyn": (40.6782, -73.9442),
    "Queens": (40.7282, -73.7949),
    "Bronx": (40.8448, -73.8648),
    "Staten Island": (40.5795, -74.1502)
}

st.subheader("Listing Information")
col1, col2 = st.columns(2)

with col1:
    neighbourhood_group = st.selectbox("Borough", list(BOROUGH_COORDS.keys()))
    room_type = st.selectbox("Room Type", ["Entire home/apt", "Private room", "Shared room"])

with col2:
    minimum_nights = st.number_input("Minimum Stay (Nights)", min_value=1, max_value=365, value=2)
    availability_365 = st.slider("Days Available per Year", 0, 365, value=180)

# Default location set based on selected borough
default_lat, default_lon = BOROUGH_COORDS[neighbourhood_group]

# Technical and historical metrics hidden in an collapsible expander
with st.expander("⚙️ Advanced Settings & Coordinates (Optional)"):
    st.caption("Adjust exact GPS coordinates or historical host metrics if available.")
    col_adv1, col_adv2 = st.columns(2)
    with col_adv1:
        latitude = st.number_input("Latitude", value=default_lat, format="%.4f")
        longitude = st.number_input("Longitude", value=default_lon, format="%.4f")
        calculated_host_listings_count = st.number_input("Host Total Listings Count", min_value=1, value=1)
    with col_adv2:
        number_of_reviews = st.number_input("Total Historical Reviews", min_value=0, value=0)
        reviews_per_month = st.number_input("Reviews per Month", min_value=0.0, value=0.0, format="%.2f")

if st.button("Predict Price", type="primary"):
    # Compute distance to Midtown Manhattan dynamically
    dist_to_midtown = np.sqrt((latitude - 40.7580)**2 + (longitude - (-73.9855))**2)

    # Input DataFrame matching model schema
    input_df = pd.DataFrame([{
        "latitude": latitude,
        "longitude": longitude,
        "minimum_nights": minimum_nights,
        "number_of_reviews": number_of_reviews,
        "reviews_per_month": reviews_per_month,
        "calculated_host_listings_count": calculated_host_listings_count,
        "availability_365": availability_365,
        "dist_to_midtown": dist_to_midtown,
        "neighbourhood_group": neighbourhood_group,
        "room_type": room_type
    }])

    pred_log = pipeline.predict(input_df)[0]
    pred_price = np.expm1(pred_log)

    st.success(f"### Estimated Price: **${pred_price:.2f}** / night")

Overwriting app.py


# Task 4

In [23]:
!pip install -q pyngrok


In [36]:
from pyngrok import ngrok
from google.colab import userdata

key= userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(key)


ngrok.kill()

public_url = ngrok.connect(8501)
print(f"🚀 Streamlit App live URL: {public_url}")

# 5. Run Streamlit in the background
!streamlit run app.py &

🚀 Streamlit App live URL: NgrokTunnel: "https://bagel-affecting-endurable.ngrok-free.dev" -> "http://localhost:8501"


2026-09-13 19:42:41.924 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.70.115.55:8501



  Stopping...
